# Notebook B — Schema drift scenario

**Pipeline:** B
**Pain Point mapped:** #1 (silent skip / no clear logs)

Reads a 5-column source CSV (customer_id, name, email, country, loyalty_tier) but writes only 4 columns to the sink — `loyalty_tier` is silently dropped. The Spark notebook itself does NOT raise a warning by default. The pipeline run will SUCCEED.

Expected SRE Agent observation: pipeline succeeded, but the App Insights customEvent `SchemaDriftDetected` (emitted at the end of this notebook) reveals 5 source columns vs 4 sink columns. The `loyalty_tier` column is missing downstream.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName('notebook-B-schema-drift').getOrCreate()

src_path = 'abfss://stage@<your-storage>.dfs.core.windows.net/customers-5col.csv'
sink_path = 'abfss://output@<your-storage>.dfs.core.windows.net/customers-4col/'

print(f'[notebook-B-schema-drift] Reading 5-column source: {src_path}')
src_df = spark.read.option('header', True).csv(src_path)
src_columns = src_df.columns
src_count = src_df.count()
print(f'[notebook-B-schema-drift] sourceColumns={len(src_columns)} sourceColumnNames={src_columns} rowsRead={src_count}')

# THE BUG: explicit mapping that drops loyalty_tier silently
sink_columns = ['customer_id', 'name', 'email', 'country']
sink_df = src_df.select(*[col(c) for c in sink_columns])
sink_df.write.mode('overwrite').parquet(sink_path)

sink_actual = spark.read.parquet(sink_path)
sink_actual_columns = sink_actual.columns
sink_actual_count = sink_actual.count()
print(f'[notebook-B-schema-drift] sinkColumns={len(sink_actual_columns)} sinkColumnNames={sink_actual_columns} rowsCopied={sink_actual_count}')

drift_detected = set(src_columns) - set(sink_actual_columns)
if drift_detected:
    print(f'[notebook-B-schema-drift] SCHEMA_DRIFT_DETECTED: dropped_columns={list(drift_detected)}')
else:
    print(f'[notebook-B-schema-drift] no schema drift')

print(f'[notebook-B-schema-drift] Status: SUCCEEDED (but drift went unreported to caller)')

## Optional — emit App Insights customEvent

If the App Insights connection string is set as a Spark conf or env var, this surfaces the drift in `customEvents` table for SRE Agent to query. Comment out the try/except if you want to test the failure mode first.

In [ ]:
import os
try:
    from opencensus.ext.azure.log_exporter import AzureEventHandler
    import logging
    conn = os.environ.get('APPINSIGHTS_CONNECTION_STRING')
    if conn:
        logger = logging.getLogger('synapse-sre-pilot')
        logger.setLevel(logging.INFO)
        logger.addHandler(AzureEventHandler(connection_string=conn))
        logger.info('SchemaDriftDetected', extra={'custom_dimensions': {
            'pipeline': 'pipeline-B-schema-drift',
            'sourceColumns': len(src_columns),
            'sinkColumns': len(sink_actual_columns),
            'droppedColumns': list(drift_detected) if drift_detected else []
        }})
        print('[notebook-B-schema-drift] customEvent SchemaDriftDetected emitted to App Insights')
    else:
        print('[notebook-B-schema-drift] APPINSIGHTS_CONNECTION_STRING not set; skipping customEvent emit')
except Exception as e:
    print(f'[notebook-B-schema-drift] customEvent emit failed (non-fatal): {e}')